In [1]:

import numpy as np


grid = [
    "S....",
    ".##..",
    "...#.",
    ".#...",
    "...#G",
]
ROWS, COLS = len(grid), len(grid[0])
start = (0, 0)
goal = (4, 4)

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
ACTION_NAMES = ["^", "v", "<", ">"]

def step(state, a):
    r, c = state
    dr, dc = ACTIONS[a]
    nr, nc = r + dr, c + dc
    # hitting a wall or obstacle: stay in place, small penalty
    if not (0 <= nr < ROWS and 0 <= nc < COLS) or grid[nr][nc] == "#":
        return state, -5, False
    if (nr, nc) == goal:
        return (nr, nc), 100, True
    return (nr, nc), -1, False          # -1 per move -> encourages short paths


Q = np.zeros((ROWS, COLS, len(ACTIONS)))
alpha = 0.1      # learning rate
gamma = 0.9      # discount factor
epsilon = 1.0    # exploration rate (decays)
episodes = 500

for ep in range(episodes):
    state = start
    for _ in range(100):  # max steps per episode
        # epsilon-greedy action choice
        if np.random.rand() < epsilon:
            a = np.random.randint(len(ACTIONS))
        else:
            a = np.argmax(Q[state])

        next_state, reward, done = step(state, a)

        # Q-learning update
        best_next = np.max(Q[next_state])
        Q[state][a] += alpha * (reward + gamma * best_next - Q[state][a])

        state = next_state
        if done:
            break
    epsilon = max(0.05, epsilon * 0.99)


state, path = start, [start]
while state != goal and len(path) < 50:
    state, _, _ = step(state, np.argmax(Q[state]))
    path.append(state)

print("Learned path:", path)
print(f"Steps: {len(path) - 1}\n")

display = [list(row) for row in grid]
for (r, c) in path[1:-1]:
    display[r][c] = "*"
for row in display:
    print(" ".join(row))

print("\nLearned policy (best action in each cell):")
for r in range(ROWS):
    line = []
    for c in range(COLS):
        if grid[r][c] == "#":
            line.append("#")
        elif (r, c) == goal:
            line.append("G")
        else:
            line.append(ACTION_NAMES[np.argmax(Q[r, c])])
    print(" ".join(line))


Learned path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]
Steps: 8

S . . . .
* # # . .
* * * # .
. # * * *
. . . # G

Learned policy (best action in each cell):
v > > > v
v # # > v
> > v # v
^ # > > v
> > ^ # G
